# Logistic Regression (Hyperparameter Tuning) - Detailed Notebook

This notebook is a detailed, step-by-step version of `Logistic_Regression_HP.py`. It implements binary classification on MNIST: **digit 7 vs not 7**.

It includes:
- custom logistic regression with gradient descent
- class imbalance handling using sample weights
- PCA dimensionality reduction
- grid search on PCA components, learning rate, and threshold
- final evaluation on a held-out test set


## 1) Imports
We import NumPy for numerical operations and scikit-learn for data loading, splitting, preprocessing, PCA, and evaluation metrics.

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

## 2) Core Logistic Regression Functions

### This project implements logistic regression from scratch (instead of using `sklearn.linear_model.LogisticRegression`) to explicitly show:
- the sigmoid computation
- gradient derivation usage
- manual gradient descent updates
- per-sample class weighting for imbalance

### Class imbalance strategy in this file
`not 7` (label `-1`) receives lower weight (`0.5`), while `7` (label `+1`) keeps full weight (`1.0`) to handle class imbalance. This decreases majority-class influence during optimization.

In [2]:
def sigmoid(z):
    """Sigmoid activation function with clipping for numerical stability."""
    z = np.clip(z, -250, 250)
    return 1.0 / (1.0 + np.exp(-z))


def calculate_gradient_pdf(W, X, y):
    """
    Weighted gradient for logistic regression in bipolar labels {-1, +1}.

    Weighting rule:
    - y == -1 (not 7): weight 0.5
    - y == +1 (7):     weight 1.0
    """
    linear_comb = X @ W
    exponent = np.clip(y * linear_comb, -250, 250)
    denominator = 1.0 + np.exp(exponent)

    # Base derivative term
    multiplier = -y / denominator

    # Class imbalance handling through sample weights
    sample_weights = np.where(y == -1, 0.5, 1.0)
    multiplier = multiplier * sample_weights

    # Aggregate weighted gradients across samples
    grad = X.T @ multiplier
    return grad


def gradient_descent_pdf(X, y, eta=1e-5, iterations=50, tol=1e-7):
    """
    Gradient descent optimization for custom logistic regression.

    Parameters:
    - X: features without bias column
    - y: labels in {-1, +1}
    - eta: learning rate
    - iterations: maximum updates
    - tol: early stop based on gradient norm
        Gradient descent update rule:
        W(n+1) = W(n) - eta * gradient
    
    """
    X_b = np.c_[np.ones((X.shape[0], 1)), X]   # add bias term by concatenating 1s withold X
    W = np.zeros(X_b.shape[1])

    for i in range(iterations):
        grad = calculate_gradient_pdf(W, X_b, y)
        W = W - eta * grad

        # Stop if updates become very small
        if np.linalg.norm(grad) < tol:
            break

        # Less frequent logging to keep output readable
        if (i + 1) % 5000 == 0:
            print(f"    Iteration {i+1}/{iterations} completed.")

    return W


def predict_pdf(X, W, threshold=0.5):
    """Predict bipolar labels {-1, +1} from learned weights."""
    X_b = np.c_[np.ones((X.shape[0], 1)), X]
    probs = sigmoid(X_b @ W)
    return np.where(probs >= threshold, 1, -1)

## 3) Load and Split MNIST

We do a two-stage split:
1. hold out **10,000** samples for the final test set
2. from the remaining data, hold out **10,000** for validation

This gives us:
- Train: used to learn weights
- Validation: used for hyperparameter selection
- Test: used only once for final unbiased performance estimate

In [3]:
print("Fetching MNIST dataset... (this may take a minute)")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

print("Preparing and splitting data...")
y_bipolar = np.where(y == '7', 1, -1)

# Split 1: reserve final test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_bipolar, test_size=10000, random_state=42
)

# Split 2: reserve validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=10000, random_state=42
)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Fetching MNIST dataset... (this may take a minute)
Preparing and splitting data...
Train shape: (50000, 784), Val shape: (10000, 784), Test shape: (10000, 784)


## 4) Standardize Features

We fit `StandardScaler` on the training set only, then transform validation and test sets.

Why this matters:
- avoids data leakage
- keeps feature scales consistent for gradient descent and PCA

In [4]:
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Scaling features...


## 5) Hyperparameter Tuning (Grid Search)

We test combinations of:
- PCA components: `[50, 100, 300]`
- learning rates: `[1e-3, 1e-2]`
- decision thresholds: `[0.40, 0.50, 0.60]`

For each combination:
1. fit PCA on training data
2. train custom logistic regression
3. evaluate validation accuracy for each threshold
4. keep the best-performing setup

In [9]:
pca_components_to_test = [50, 100, 200, 300]
learning_rates_to_test = [1e-3, 1e-2]
thresholds_to_test = [0.40, 0.50]
iterations_to_test =  [10000]

best_val_accuracy = 0.0
best_params = {}
best_W = None
best_pca = None

print("\nStarting automated hyperparameter tuning...")
print("==========================================")

for n_components in pca_components_to_test:
    for eta in learning_rates_to_test:
        for iterations in iterations_to_test:
            print(f"Testing combination -> PCA Components: {n_components} | Learning Rate: {eta} | Iterations: {iterations}")

            pca = PCA(n_components=n_components, random_state=42)
            X_train_features = pca.fit_transform(X_train_scaled)
            X_val_features = pca.transform(X_val_scaled)

            W_hat = gradient_descent_pdf(
                X_train_features,
                y_train,
                eta=eta,
                iterations=iterations
            )

            # Tune threshold on the validation set for this trained model
            local_best_acc = -1.0
            local_best_threshold = None

            for threshold in thresholds_to_test:
                y_pred_val = predict_pdf(X_val_features, W_hat, threshold=threshold)
                val_acc = accuracy_score(y_val, y_pred_val)
                print(f"    Threshold {threshold:.2f} -> Validation Accuracy: {val_acc * 100:.2f}%")

                if val_acc > local_best_acc:
                    local_best_acc = val_acc
                    local_best_threshold = threshold

            print(f"  -> Best threshold for this combo: {local_best_threshold:.2f} | Accuracy: {local_best_acc * 100:.2f}\n")

            if local_best_acc > best_val_accuracy:
                best_val_accuracy = local_best_acc
                best_params = {'n_components': n_components, 'eta': eta, 'threshold': local_best_threshold, 'iterations': iterations}
                best_W = W_hat
                best_pca = pca

print("==========================================")
print("TUNING COMPLETE!")
print(f"Best Validation Accuracy : {best_val_accuracy * 100:.2f}%")
print(f"Best Parameters Found    : PCA = {best_params['n_components']}, eta = {best_params['eta']}, threshold = {best_params['threshold']:.2f}, iterations = {best_params['iterations']}")
print("==========================================")


Starting automated hyperparameter tuning...
Testing combination -> PCA Components: 50 | Learning Rate: 0.001 | Iterations: 10000
    Iteration 5000/10000 completed.
    Iteration 10000/10000 completed.
    Threshold 0.40 -> Validation Accuracy: 97.92%
    Threshold 0.50 -> Validation Accuracy: 97.91%
  -> Best threshold for this combo: 0.40 | Accuracy: 97.92

Testing combination -> PCA Components: 50 | Learning Rate: 0.01 | Iterations: 10000
    Iteration 5000/10000 completed.
    Iteration 10000/10000 completed.
    Threshold 0.40 -> Validation Accuracy: 98.06%
    Threshold 0.50 -> Validation Accuracy: 98.07%
  -> Best threshold for this combo: 0.50 | Accuracy: 98.07

Testing combination -> PCA Components: 100 | Learning Rate: 0.001 | Iterations: 10000
    Iteration 5000/10000 completed.
    Iteration 10000/10000 completed.
    Threshold 0.40 -> Validation Accuracy: 97.82%
    Threshold 0.50 -> Validation Accuracy: 97.80%
  -> Best threshold for this combo: 0.40 | Accuracy: 97.82

T

## 6) Final Evaluation on Test Set

After selecting the best hyperparameters on validation data, we evaluate once on the unseen test set.

Reported metrics:
- accuracy
- precision (for class `7`)
- recall (for class `7`)
- F1-score (for class `7`)
- confusion matrix

In [10]:
print("\nEvaluating winning model on the unseen Test Set...")

X_test_features = best_pca.transform(X_test_scaled)
X_train_features_best = best_pca.transform(X_train_scaled)

best_threshold = best_params['threshold']
y_pred_train = predict_pdf(X_train_features_best, best_W, threshold=best_threshold)
y_pred_test = predict_pdf(X_test_features, best_W, threshold=best_threshold)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test, pos_label=1)
recall = recall_score(y_test, y_pred_test, pos_label=1)
f1 = f1_score(y_test, y_pred_test, pos_label=1)
cm = confusion_matrix(y_test, y_pred_test, labels=[1, -1])

print(f"\nTraining Accuracy : {train_acc * 100:.2f}%")
print(f"Testing Accuracy  : {test_acc * 100:.2f}%")
print(f"Precision         : {precision * 100:.2f}%")
print(f"Recall            : {recall * 100:.2f}%")
print(f"F1-Score          : {f1 * 100:.2f}%")
print(f"Threshold Used     : {best_threshold:.2f}")

print("\nConfusion Matrix (Test Set)")
print("Rows = Actual, Columns = Predicted")
print("Order of classes: [7 (+1), Not 7 (-1)]")
print(cm)


Evaluating winning model on the unseen Test Set...

Training Accuracy : 98.24%
Testing Accuracy  : 98.25%
Precision         : 90.00%
Recall            : 93.84%
F1-Score          : 91.88%
Threshold Used     : 0.50

Confusion Matrix (Test Set)
Rows = Actual, Columns = Predicted
Order of classes: [7 (+1), Not 7 (-1)]
[[ 990   65]
 [ 110 8835]]


## 7) Interpretation Notes

- **Weighted training**: This notebook reduces the majority class contribution by using a lower weight for `not 7`.
- **PCA tradeoff**: Lower dimensions can speed training but may lose useful signal; tuning checks this tradeoff.
- **Validation vs Test**: Hyperparameters are selected on validation only; test is kept untouched for honest final reporting.
- **Potential next improvements**:
  - compare against undersampling and oversampling
  - add ROC-AUC / PR-AUC for imbalanced-data diagnostics